# Phase 1 — LLM corpus generation pilot
Generate **10** sample responses with Qwen2.5-0.5B-Instruct and measure generation speed. This is a feasibility test, **not** the final 20K-document dataset.

Select **Runtime → Change runtime type → T4 GPU** if available. CPU also works but can be much slower.


In [ ]:
%pip -q install 'transformers>=4.44,<5' accelerate safetensors


In [ ]:
import json, os, platform, statistics, time
from datetime import datetime, timezone
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_NEW_TOKENS = 128
SEED = 42
torch.manual_seed(SEED)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU)')

start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype='auto', device_map='auto')
model.eval()
load_seconds = time.perf_counter() - start
print(f'Model load: {load_seconds:.1f} s')


In [ ]:
PROMPTS = [
    'Explain how transformer attention works in simple terms.',
    'Describe three practical cybersecurity defenses for a small business.',
    'Explain the causes of inflation with an everyday example.',
    'How does a lithium-ion battery work in an electric vehicle?',
    'Compare supervised and unsupervised machine learning.',
    'What causes earthquakes and how are they measured?',
    'Describe the main benefits and challenges of renewable energy.',
    'Explain how relational database indexes speed up queries.',
    'How do vaccines train the immune system?',
    'Describe how a recommendation system suggests movies.'
]
assert len(PROMPTS) == len(set(PROMPTS))

def generate(prompt):
    messages = [{'role':'system','content':'Give a clear, factual answer in 80-130 words.'}, {'role':'user','content':prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([rendered], return_tensors='pt').to(model.device)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    generated = out[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), int(generated.numel()), elapsed

results = []
for i, prompt in enumerate(PROMPTS, 1):
    answer, tokens, seconds = generate(prompt)
    results.append({'id':i,'prompt':prompt,'response':answer,'generated_tokens':tokens,'generation_seconds':round(seconds,4),'tokens_per_second':round(tokens/seconds,2) if seconds else None,'model':MODEL_ID})
    print(f'{i:02d}/{len(PROMPTS)} | {seconds:.2f}s | {tokens} tokens | {tokens/seconds:.1f} tok/s')

median_s = statistics.median(r['generation_seconds'] for r in results)
total_s = sum(r['generation_seconds'] for r in results)
print(f'\nMedian per response: {median_s:.2f}s | Pilot total generation: {total_s:.1f}s')
for n in [5000, 10000, 20000]:
    hours = median_s * n / 3600
    print(f'Projected {n:,} responses: {hours:.1f} h (sequential; excludes model loading, retries, interruptions)')


In [ ]:
# Save measured pilot artifacts to Google Drive (no 20K generation is triggered).
from google.colab import drive
drive.mount('/content/drive')
out_dir = Path('/content/drive/MyDrive/HP-SS-Engine/data/raw')
out_dir.mkdir(parents=True, exist_ok=True)
with (out_dir/'qwen_pilot_10.jsonl').open('w', encoding='utf-8') as f:
    for item in results: f.write(json.dumps(item, ensure_ascii=False)+'\n')
summary = {'timestamp_utc':datetime.now(timezone.utc).isoformat(),'model':MODEL_ID,'n':len(results),'max_new_tokens':MAX_NEW_TOKENS,'seed':SEED,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'load_seconds':load_seconds,'median_generation_seconds':median_s,'total_generation_seconds':total_s,'projected_hours_by_n':{str(n):median_s*n/3600 for n in [5000,10000,20000]},'notes':'Sequential pilot; no claim of full-corpus feasibility until real run.'}
(out_dir/'qwen_pilot_10_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:',out_dir/'qwen_pilot_10.jsonl')
print('Saved:',out_dir/'qwen_pilot_10_summary.json')
